# Расширенный датасет всех объектов недвижимости

В результат попадают все неудалённые объекты `nedv_ul_and_ip`. Если подтверждённая связь с договором найдена, договорные поля заполняются. Если связь не найдена, объект остаётся в результате, а договорные поля остаются пустыми.


In [29]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [31]:
PROJECT_ROOT = Path.cwd()  # или Path('/полный/путь/к/проекту')
OUTPUT_DIR = PROJECT_ROOT / 'данные'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)

Корень проекта: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python


# 1. Подключение к Сфере


In [32]:
CREDENTIALS_PATH = PROJECT_ROOT / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')

Учётные данные прочитаны, подключение к Сфере создано


In [33]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

,database_name,user_name
0,postgres,svetovavs


# 2. Подключение к Oracle КХД



In [34]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


Подключение к КХД создано


In [35]:
with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = [row[0] for row in cursor.fetchall()]

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Доступные таблицы:', available_khd_tables)

expected_khd_tables = {'EGRN_DATA'}
missing_khd_tables = sorted(expected_khd_tables - set(available_khd_tables))
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

Пользователь КХД: SVETOVAVS
База КХД: ODSPROD
Доступные таблицы: ['BUILDINGS', 'EGRN_DATA']


# 3. SQL Сфера, расширенный подход


In [36]:
expanded_sql = r"""
/*
запускать в сфере

запрос собирает все неудаленные объекты недвижимости
если объект связан с подходящим договором данные договора заполняются
если связь не найдена объект остается в результате с пустыми полями договора

одна строка для связанного объекта означает объект в одном договоре
одна строка для несвязанного объекта означает его последнюю версию характеристик
*/

with task_candidates as (
    /* отбираем подходящие задачи оформления */
    select
        t.id as task_id,
        r.id as request_id,
        c.id as contract_id,
        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* оставляем последнюю подходящую задачу каждого договора */
    select
        task_id,
        request_id,
        contract_id
    from task_candidates
    where task_number = 1
),

linked_object_candidates as (
    /* находим недвижимость в выбранных задачах */
    select
        ch.insurance_object_id as object_id,
        ch.id as characteristics_id,
        link.id as task_object_link_id,
        selected.task_id,
        selected.request_id,
        selected.contract_id,
        row_number() over (
            partition by selected.task_id, ch.insurance_object_id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc
        ) as link_number
    from selected_tasks selected
    join bps_request_ins_task_insurance_object link
        on link.parent_id = selected.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_links as (
    /* убираем повторные связи одного объекта с одной задачей */
    select
        object_id,
        characteristics_id,
        task_object_link_id,
        task_id,
        request_id,
        contract_id
    from linked_object_candidates
    where link_number = 1
),

object_versions as (
    /* нумеруем версии характеристик каждого объекта */
    select
        obj.id as object_id,
        ch.id as characteristics_id,
        row_number() over (
            partition by obj.id
            order by
                ch.version_is_active desc nulls last,
                ch.version_number desc nulls last,
                ch.version_start_date desc nulls last,
                ch.id desc nulls last
        ) as version_number
    from base_insurance_object obj
    left join base_insurance_object_characteristics ch
        on ch.insurance_object_id = obj.id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

dataset_keys as (
    /* сохраняем все найденные связи с договорами */
    select
        linked.object_id,
        linked.characteristics_id,
        linked.task_object_link_id,
        linked.task_id,
        linked.request_id,
        linked.contract_id,
        'linked'::text as row_source
    from selected_links linked

    union all

    /* добавляем объекты для которых подходящий договор не найден */
    select
        version.object_id,
        version.characteristics_id,
        null::integer as task_object_link_id,
        null::integer as task_id,
        null::integer as request_id,
        null::integer as contract_id,
        'not_linked'::text as row_source
    from object_versions version
    where version.version_number = 1
      and not exists (
          select 1
          from selected_links linked
          where linked.object_id = version.object_id
      )
),

object_link_profile as (
    /* считаем со сколькими договорами связан объект */
    select
        object_id,
        count(distinct contract_id) as contract_count
    from selected_links
    group by object_id
),

selected_characteristics as (
    /* ограничиваем расчет условий версиями из итоговой выборки */
    select distinct characteristics_id
    from dataset_keys
    where characteristics_id is not null
),

condition_summary as (
    /* сворачиваем варианты условий в одну строку */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        count(cond.insured_sum) as filled_insured_sum_count,
        count(distinct cond.insured_sum) filter (
            where cond.insured_sum is not null
        ) as distinct_insured_sum_count,
        min(cond.insured_sum) as minimum_insured_sum,
        max(cond.insured_sum) as maximum_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        array_agg(
            distinct cond.terms_option_number
            order by cond.terms_option_number
        ) filter (
            where cond.terms_option_number is not null
        ) as terms_option_numbers,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        case
            when count(distinct cond.insured_sum) filter (
                where cond.insured_sum is not null
            ) = 1
             and count(distinct cond.insured_sum_currency) filter (
                where cond.insured_sum_currency is not null
            ) <= 1
            then max(cond.insured_sum)
        end as insured_sum
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

raw_result as (
/* собираем исходные поля расширенного датасета */
select
    /* качество строки */
    keys.row_source,
    case
        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'
        when profile.contract_count = 1 then 'linked'
        else 'multiple_contracts'
    end as contract_link_status,
    coalesce(profile.contract_count, 0) as contract_count,
    (keys.contract_id is not null) as has_contract,
    (address.id is not null) as has_address,
    (conditions.insured_sum is not null) as has_target,
    case
        when conditions.condition_count is null then 'no_conditions'
        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'
        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'
        when conditions.currency_count > 1 then 'several_currencies'
        when conditions.insured_sum <= 0 then 'target_is_not_positive'
        else 'target_is_usable'
    end as target_status,

    /* идентификаторы */
    keys.object_id,
    keys.characteristics_id,
    keys.task_object_link_id,
    keys.task_id,
    keys.request_id,
    keys.contract_id,
    obj.geo_address_id,
    contract.contractor_id as policyholder_id,
    request.corporate_crm_id,

    /* целевая страховая сумма */
    conditions.insured_sum,
    conditions.insured_sum_currency,
    conditions.condition_count,
    conditions.filled_insured_sum_count,
    conditions.distinct_insured_sum_count,
    conditions.minimum_insured_sum as condition_min_insured_sum,
    conditions.maximum_insured_sum as condition_max_insured_sum,
    conditions.currency_count as condition_currency_count,
    conditions.terms_option_numbers,

    /* контрольные суммы */
    task_link.insured_sum as task_object_insured_sum,
    task_link.insured_sum_currency as task_object_insured_sum_currency,
    task.total_ins_contract_amount as contract_insured_sum,
    task.curr_ins_contract_amount as contract_amount_currency,
    task.total_ins_contract_premium as contract_premium,
    ch.insurance_value,
    ch.insurance_value_currency,
    ch.insurance_value_basis,
    ch.pledged_value,
    conditions.minimum_per_occurrence_limit,
    conditions.maximum_per_occurrence_limit,

    /* объект */
    obj.obj_type as object_type,
    obj.elementary_obj_type,
    obj.obj_name as object_name,
    obj.description as object_description,
    obj.original_address,
    obj.active as object_is_active,
    obj.d_create as object_create_date,
    obj.d_change as object_change_date,

    /* характеристики объекта */
    ch.version_number as characteristics_version_number,
    ch.version_start_date as characteristics_version_start_date,
    ch.version_end_date as characteristics_version_end_date,
    ch.version_is_active as characteristics_version_is_active,
    ch.ownership_type,
    ch.is_pledged,
    ch.is_leased,
    ch.insured_components,
    ch.activity_types,
    ch.risk_natures,
    ch.insurance_territory,
    ch.has_losses,
    ch.insurance_object_loss_history,
    ch.characteristics ->> 'total_area_sq_m' as total_area,
    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
    ch.characteristics ->> 'construction_year' as construction_year,
    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,
    ch.characteristics ->> 'total_floors_count' as floors_count,
    ch.characteristics ->> 'occupied_floor' as occupied_floor,
    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,
    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,
    ch.characteristics ->> 'roofing_material' as roofing_material,
    ch.characteristics ->> 'fire_alarm_system_availability'
        as fire_alarm_system_availability,
    ch.characteristics ->> 'fire_suppression_system_availability'
        as fire_suppression_system_availability,
    ch.characteristics ->> 'nearest_fire_station_distance_km'
        as nearest_fire_station_distance_km,
    ch.characteristics as object_characteristics_json,

    /* адрес */
    address.full_address,
    address.postal_code,
    address.region_id as address_region_id,
    address.area as district,
    address.settlement_type,
    address.settlement,
    address.street_type,
    address.street,
    address.house,
    address.building,
    address.block,
    address.flat,
    address.office,
    address.fias_code,
    address.longitude,
    address.latitude,
    address.address_dgis_id,

    /* договор */
    contract.n_contract as contract_number,
    contract.document_status as contract_status,
    contract.system_type as contract_source_system,
    contract.ins_product_sbs as insurance_product,
    contract.d_sign_contract as contract_sign_date,
    contract.d_start_contract as contract_start_date,
    contract.d_end_contract as contract_end_date,
    contract.prevcontract_id as previous_contract_id,
    contract.rootcontract_id as root_contract_id,
    previous_contract.n_contract as previous_contract_number,
    previous_contract.d_start_contract as previous_contract_start_date,
    previous_contract.d_end_contract as previous_contract_end_date,

    /* задача и заявка */
    task.task_type,
    task.status as task_status,
    task.ins_document_type,
    task.ins_refuse,
    task.d_create as task_create_date,
    task.d_conclusion_ins_contract as contract_conclusion_date,
    task.ins_product as task_product,
    task.industry as task_industry,
    task.subindustry as task_subindustry,
    task.locations_count,
    task.multi_location,
    task.object_description as task_object_description,
    request.business_segment,
    request.sale_channel,
    request.ins_product as request_product,

    /* страхователь и crm */
    policyholder.inn as policyholder_inn,
    policyholder.company_name_short as policyholder_name,
    policyholder.cdi_id as policyholder_cdi_id,
    crm.segment as crm_segment,
    crm.macroindustry as crm_macroindustry,
    crm.industry as crm_industry,
    crm.okved as crm_okved,

    /* дата состояния строки */
    coalesce(
        task.d_conclusion_ins_contract::timestamp with time zone,
        contract.d_sign_contract,
        ch.version_start_date,
        obj.d_create
    ) as as_of_date
from dataset_keys keys
join base_insurance_object obj
    on obj.id = keys.object_id
left join base_insurance_object_characteristics ch
    on ch.id = keys.characteristics_id
left join condition_summary conditions
    on conditions.characteristics_id = keys.characteristics_id
left join bps_request_ins_task_insurance_object task_link
    on task_link.id = keys.task_object_link_id
left join bps_request_ins_task task
    on task.id = keys.task_id
left join bps_request_ins request
    on request.id = keys.request_id
left join bps_contract contract
    on contract.id = keys.contract_id
left join bps_contract previous_contract
    on previous_contract.id = contract.prevcontract_id
left join bps_contractor policyholder
    on policyholder.id = contract.contractor_id
left join bps_corporate_crm crm
    on crm.id = request.corporate_crm_id
left join base_geo_address address
    on address.id = obj.geo_address_id
left join object_link_profile profile
    on profile.object_id = keys.object_id
),

standardized_result as (
    /* приводим результат к общей структуре двух датасетов */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    has_contract desc,
    as_of_date desc nulls last,
    object_id;

"""


In [37]:
with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(expanded_sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
display(expanded_df.head(3))

Строк: 14441
Колонок: 102


,row_source,contract_link_status,contract_count,has_contract,has_address,has_target,target_status,contract_id,contract_number,previous_contract_id,root_contract_id,request_id,task_id,task_object_link_id,characteristics_id,object_id,geo_address_id,policyholder_id,corporate_crm_id,as_of_date,contract_conclusion_date,contract_sign_date,contract_start_date,contract_end_date,contract_status,ins_document_type,insurance_product,real_estate_objects_in_contract,object_name,object_description,object_type,elementary_obj_type,total_area,occupied_area,construction_year,capital_repair_year,floors_count,occupied_floor,walls_material,overlap_material,roofing_material,ownership_type,is_leased,insured_components,activity_types,risk_natures,insurance_territory,full_address,original_address,postal_code,...,settlement,street,house,building,block,flat,office,fias_code,longitude,latitude,address_dgis_id,policyholder_inn,policyholder_name,policyholder_cdi_id,crm_segment,crm_macroindustry,crm_industry,crm_okved,business_segment,task_industry,task_subindustry,contract_insured_sum,contract_amount_currency,task_object_insured_sum,task_object_insured_sum_currency,condition_min_insured_sum,condition_max_insured_sum,insured_sum,insured_sum_currency,condition_currency_count,contract_premium,insurance_value,insurance_value_currency,insurance_value_basis,is_pledged,pledged_value,minimum_per_occurrence_limit,maximum_per_occurrence_limit,previous_contract_number,previous_contract_start_date,previous_contract_end_date,condition_count,characteristics_version_number,characteristics_version_start_date,characteristics_version_end_date,characteristics_version_is_active,object_characteristics_json,task_type,task_status,ins_refuse
0,linked,linked,1,True,True,True,target_is_usable,117289499.0,013ПР4040041101,None,None,438756.0,692897.0,32072.0,18539,18561,11297.0,27250513.0,117888.0,2026-08-23 21:00:00+00:00,2026-08-24,2025-09-14 21:00:00+00:00,2025-09-11 21:00:00+00:00,2026-09-11 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,Имущество юридических лиц (залоги) - сверхлими...,1.0,other,офисное помещение,prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,[interior_decoration],[],[],None,"Москва, Братиславская улица, 16 к1",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,37.758072,55.659104,4504235282536969,7723627614,"АО ""ТОРГОВЫЙ ДОМ ТРАКТ""",None,ksb,NaN,NaN,NaN,ksb,estate,other_commercial_real_estate,6.000000e+07,RUB,NaN,NaN,35000000.0,35000000.0,35000000.0,RUB,1.0,42000.0,NaN,NaN,,False,None,NaN,NaN,None,None,None,1.0,1,2026-06-19 12:59:15.616472+00:00,None,True,{},draft_contract,operational_archive,False
1,linked,linked,1,True,True,True,target_is_usable,115920262.0,013БС4040040730,None,None,433282.0,695918.0,34585.0,21134,21156,12226.0,53538753.0,244208.0,2026-08-17 21:00:00+00:00,2026-08-18,2025-08-20 21:00:00+00:00,2025-08-17 21:00:00+00:00,2026-08-17 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,7.0,other,"Здание АТС 650, 694 5-ти этаж. с подвалом кирп...",prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,[None],[],[],None,"Москва, улица Малая Дмитровка, 5/9 / Москва, Н...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,37.605630,55.767954,4504235282745299,7723625776,NaN,None,ksb,estate,NaN,NaN,ksb,estate,bc,3.568906e+09,RUB,NaN,NaN,685254000.0,685254000.0,685254000.0,RUB,1.0,296219.2,685254000.0,RUB,685 254 000,False,None,NaN,NaN,None,None,None,1.0,1,2026-07-03 13:45:32.831145+00:00,None,True,{},draft_contract,operational_archive,False
2,linked,linked,1,True,True,True,target_is_usable,115920262.0,013БС4040040730,None,None,433282.0,695918.0,34586.0,21135,21157,12227.0,53538753.0,244208.0,2026-08-17 21:00:00+00:00,2026-08-18,2025-08-20 21:00:00+00:00,2025-08-17 21:00:00+00:00,2026-08-17 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,7.0,other,"ЗДАНИЕ АТС 252,255 КИРПИЧНОЕ",prop_ass

# 4. Проверка заполненности


In [38]:
required_columns = {
    'row_source', 'object_id', 'characteristics_id', 'contract_id',
    'elementary_obj_type', 'has_contract', 'has_address', 'has_target',
    'target_status'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Уникальных характеристик',
        'Уникальных договоров',
        'Строк с договором',
        'Строк с адресом',
        'Строк с target',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['characteristics_id'].nunique(dropna=True),
        expanded_df['contract_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).astype(bool).sum(),
        expanded_df['has_address'].fillna(False).astype(bool).sum(),
        expanded_df['has_target'].fillna(False).astype(bool).sum(),
        expanded_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

,Показатель,Значение
0,Строк,14441
1,Уникальных объектов,14431
2,Уникальных характеристик,14431
3,Уникальных договоров,558
4,Строк с договором,1187
5,Строк с адресом,11488
6,Строк с target,13271
7,Строк с пустым типом объекта,0


In [39]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))
display(expanded_df['target_status'].fillna('empty').value_counts(dropna=False))

row_source
not_linked    13254
linked         1187
Name: count, dtype: int64

elementary_obj_type
nedv_ul_and_ip    14441
Name: count, dtype: int64

target_status
target_is_usable          13229
target_is_empty             675
no_conditions               451
several_target_values        44
target_is_not_positive       42
Name: count, dtype: int64

# 5. Соединение с ЕГРН

В Oracle передаются только технический номер строки, ID объекта, адрес и площадь. Поиск выполняется на уровне здания. Данные ЕГРН присоединяются только при одном кандидате; неоднозначные случаи остаются пустыми.


In [ ]:
# передаём в Oracle только поля, которые нужны для поиска ЕГРН
sphere_with_row_id = expanded_df.copy()
sphere_with_row_id.insert(0, 'sphere_row_id', range(1, len(sphere_with_row_id) + 1))

stage_columns = [
    'sphere_row_id',
    'contract_id',
    'contract_number',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'real_estate_objects_in_contract',
    'object_description',
    'total_area',
    'full_address',
    'original_address',
    'postal_code',
    'settlement',
    'street',
    'house',
    'building',
    'block',
    'flat',
    'office',
]

missing_stage_columns = [
    column for column in stage_columns
    if column not in sphere_with_row_id.columns
]
if missing_stage_columns:
    raise ValueError(
        'Для поиска ЕГРН не хватает колонок: '
        + ', '.join(missing_stage_columns)
    )

def oracle_text(value):
    if value is None or pd.isna(value):
        return None
    return str(value)

stage_rows = [
    tuple(oracle_text(value) for value in row)
    for row in sphere_with_row_id[stage_columns].itertuples(
        index=False,
        name=None,
    )
]

# приватная таблица существует только в текущем подключении Oracle
khd_connection.rollback()
with khd_connection.cursor() as cursor:
    try:
        cursor.execute('drop table ORA$PTT_SPHERE_EGRN_KEYS')
    except oracledb.DatabaseError as error:
        if error.args[0].code != 942:
            raise

    cursor.execute(
        """
        create private temporary table ORA$PTT_SPHERE_EGRN_KEYS (
            sphere_row_id number not null,
            contract_id varchar2(200),
            contract_number varchar2(500),
            task_id varchar2(200),
            task_object_link_id varchar2(200),
            characteristics_id varchar2(200),
            object_id varchar2(200),
            geo_address_id varchar2(200),
            real_estate_objects_in_contract varchar2(200),
            object_description varchar2(4000),
            total_area varchar2(200),
            full_address varchar2(4000),
            original_address varchar2(4000),
            postal_code varchar2(100),
            settlement varchar2(1000),
            street varchar2(1000),
            house varchar2(500),
            building varchar2(500),
            block varchar2(500),
            flat varchar2(500),
            office varchar2(500)
        ) on commit preserve definition
        """
    )

    cursor.executemany(
        """
        insert into ORA$PTT_SPHERE_EGRN_KEYS (
            sphere_row_id, contract_id, contract_number, task_id,
            task_object_link_id, characteristics_id, object_id,
            geo_address_id, real_estate_objects_in_contract,
            object_description, total_area, full_address,
            original_address, postal_code, settlement, street,
            house, building, block, flat, office
        ) values (
            :1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11,
            :12, :13, :14, :15, :16, :17, :18, :19, :20, :21
        )
        """,
        stage_rows,
    )

print('Во временную таблицу Oracle передано строк:', len(stage_rows))


In [ ]:
egrn_sql = r"""
/*
Соединение временной выборки объектов Сферы с ЕГРН по адресу.

Запускать в Oracle.

Результат содержит одну строку на объект Сферы. Данные ЕГРН заполняются
только тогда, когда по адресу найдена одна запись уровня здания.

Для поиска используются населённый пункт, улица и дом. Корпус и строение
учитываются, если они указаны.
Индекс и регион сравниваются, если они заполнены с обеих сторон.
Если по адресу найдено несколько зданий, площадь используется как
дополнительная проверка. Если площади нет или она не помогла, кандидаты
по адресу не отсекаются.

В поиск попадают только типы ЕГРН «здание», «сооружение» и «строение»
с уровнем адреса FIAS_HOUSE. Квартиры, офисы, комнаты и помещения исключены.
Если по одному адресу Сферы записано несколько страховых объектов,
одно найденное здание ЕГРН присоединяется к каждому из них.

Источник — приватная временная таблица ORA$PTT_SPHERE_EGRN_KEYS.
Её заполняет notebook перед выполнением этого запроса.
*/

with sphere_source as (
    /* Берём только поля, которые нужны для проверки соединения. */
    select /*+ materialize */
        s.sphere_row_id,
        s.contract_id,
        s.contract_number,
        s.task_id,
        s.task_object_link_id,
        s.characteristics_id,
        s.object_id,
        s.geo_address_id,
        s.real_estate_objects_in_contract,
        cast(s.object_description as varchar2(4000))
            as object_description,
        s.total_area,
        cast(s.full_address as varchar2(4000)) as full_address,
        cast(s.original_address as varchar2(4000)) as original_address,
        s.postal_code,
        s.settlement,
        s.street,
        s.house,
        s.building,
        s.block,
        s.flat,
        s.office
    from ORA$PTT_SPHERE_EGRN_KEYS s
),

sphere_text as (
    /* Если нормализованного адреса нет, используем адрес, введённый вручную. */
    select
        s.*,
        replace(
            regexp_replace(trim(s.total_area), '[[:space:]]+', ''),
            ',',
            '.'
        ) as sphere_area_text,
        coalesce(
            nullif(trim(s.full_address), ''),
            nullif(trim(s.original_address), '')
        ) as source_address,
        regexp_replace(
            regexp_replace(
                replace(
                    lower(
                        replace(
                            coalesce(
                                nullif(trim(s.full_address), ''),
                                nullif(trim(s.original_address), '')
                            ),
                            chr(160),
                            ' '
                        )
                    ),
                    'ё',
                    'е'
                ),
                '[;|]+',
                ','
            ),
            '[[:space:]]*,[[:space:]]*',
            ', '
        ) as address_text
    from sphere_source s
),

sphere_parts_raw as (
    /* Берём готовые части адреса, а при их отсутствии разбираем полный адрес. */
    select
        s.*,
        coalesce(
            nullif(trim(s.postal_code), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{6})([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as postal_code_raw,
        regexp_substr(
            s.address_text,
            '(^|,)[[:space:]]*([^,]*(область|обл[.]?|край|республика|респ[.]?)[^,]*)',
            1, 1, 'i', 2
        ) as region_raw,
        coalesce(
            nullif(trim(s.settlement), ''),
            regexp_substr(
                s.address_text,
                '^[[:space:]]*([^,(]+)[[:space:]]*[(]',
                1, 1, 'i', 1
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(город|г)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)[.]?[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]*,[[:space:]]*([^,]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея))([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as locality_raw,
        coalesce(
            nullif(trim(s.street), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея)[[:space:]]+([^,]+)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([^,]+)[[:space:]]+(улица|ул[.]?|проспект|пр-кт|переулок|пер[.]?|шоссе|ш[.]?|набережная|наб[.]?|бульвар|б-р|проезд|площадь|пл[.]?|тракт|аллея)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as street_raw,
        coalesce(
            nullif(trim(s.house), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(дом|д)[.]?[[:space:]]*([0-9]+[а-яa-z]?([/-][0-9а-яa-z]+)?)',
                1, 1, 'i', 3
            ),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*([0-9]{1,5}[а-яa-z]?([/-][0-9а-яa-z]+)?)([[:space:]]*,|$)',
                1, 1, 'i', 2
            )
        ) as house_raw,
        coalesce(
            nullif(trim(s.block), ''),
            regexp_substr(
                s.address_text,
                '(^|,)[[:space:]]*(корпус|корп|к)([.]|[[:space:]])+[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 4
            )
        ) as korpus_raw,
        coalesce(
            nullif(trim(s.building), ''),
            regexp_substr(
                s.address_text,
                '(^|,|[[:space:]])(строение|стр)[.]?[[:space:]]*([0-9а-яa-z/-]+)',
                1, 1, 'i', 3
            )
        ) as stroenie_raw
    from sphere_text s
),

sphere_prepared as (
    /* Приводим части адреса к одному виду для сравнения. */
    select /*+ materialize */
        s.*,
        case
            when regexp_like(
                s.sphere_area_text,
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                s.sphere_area_text,
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area,
        regexp_replace(s.postal_code_raw, '[^0-9]+', '')
            as sphere_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.region_raw)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.locality_raw)), 'ё', 'е'),
                '(^|[[:space:]])(город|г|пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_locality,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(s.street_raw)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as sphere_street,
        regexp_replace(
            replace(lower(trim(s.house_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_house,
        regexp_replace(
            replace(lower(trim(s.korpus_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_korpus,
        regexp_replace(
            replace(lower(trim(s.stroenie_raw)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as sphere_stroenie
    from sphere_parts_raw s
),

sphere_core_keys as (
    /* Короткий список адресов ограничивает поиск по большой таблице ЕГРН. */
    select distinct
        sphere_locality,
        sphere_street,
        sphere_house
    from sphere_prepared
    where sphere_locality is not null
      and sphere_street is not null
      and sphere_house is not null
),

egrn_normalized as (
    /* Готовим адрес и минимальный набор данных ЕГРН. */
    select /*+ materialize */
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || to_char(e.cad_ind)
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date,
        regexp_replace(trim(e.postal_code), '[^0-9]+', '')
            as egrn_postal_code,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.region)), 'ё', 'е'),
                '(^|[[:space:]])(область|обл|край|республика|респ)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_region,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.city)), 'ё', 'е'),
                '(^|[[:space:]])(город|г)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_city,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.settlement)), 'ё', 'е'),
                '(^|[[:space:]])(пгт|поселок городского типа|рабочий поселок|р[.]?п|поселок|пос|село|с|деревня|д|хутор|х)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_settlement,
        regexp_replace(
            regexp_replace(
                replace(lower(trim(e.street)), 'ё', 'е'),
                '(^|[[:space:]])(улица|ул|проспект|пр-кт|переулок|пер|шоссе|ш|набережная|наб|бульвар|б-р|проезд|площадь|пл|тракт|аллея)([.]|[[:space:]]|$)',
                ' '
            ),
            '[^[:alnum:]]+',
            ''
        ) as egrn_street,
        regexp_replace(
            replace(lower(trim(e.house_number)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_house,
        regexp_replace(
            replace(lower(trim(e.vladenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_vladenie,
        regexp_replace(
            replace(lower(trim(e.korpus)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_korpus,
        regexp_replace(
            replace(lower(trim(e.stroenie)), 'ё', 'е'),
            '[^[:alnum:]]+',
            ''
        ) as egrn_stroenie
    from DM_RISK_AVATAR.EGRN_DATA e
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) in (
          'здание',
          'сооружение',
          'строение'
      )
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (
          e.cadaster is not null
          or e.cad_ind is not null
      )
),

egrn_candidates as (
    /* Оставляем только адреса, которые могут относиться к нашей выгрузке. */
    select e.*
    from egrn_normalized e
    join sphere_core_keys k
        on k.sphere_street = e.egrn_street
       and k.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and k.sphere_locality in (e.egrn_city, e.egrn_settlement)
),

address_matches as (
    /* Сравниваем адрес только до уровня здания. */
    select
        s.sphere_row_id,
        s.sphere_area,
        e.*
    from sphere_prepared s
    join egrn_candidates e
       on s.sphere_street = e.egrn_street
       and s.sphere_house in (e.egrn_house, e.egrn_vladenie)
       and s.sphere_locality in (e.egrn_city, e.egrn_settlement)
       and (
           s.sphere_region is null
           or e.egrn_region is null
           or s.sphere_region = e.egrn_region
       )
       and (
           s.sphere_postal_code is null
           or e.egrn_postal_code is null
           or s.sphere_postal_code = e.egrn_postal_code
       )
       and (
           s.sphere_korpus is null
           or s.sphere_korpus = e.egrn_korpus
       )
       and (
           s.sphere_stroenie is null
           or s.sphere_stroenie = e.egrn_stroenie
       )
    where s.sphere_locality is not null
      and s.sphere_street is not null
      and s.sphere_house is not null
),

ranked_egrn_rows as (
    /* Один кадастровый объект может повторяться. Оставляем свежую запись. */
    select
        m.*,
        row_number() over (
            partition by m.sphere_row_id, m.egrn_key
            order by
                m.row_update_date desc nulls last,
                m.ias_update_date desc nulls last,
                m.cad_ind desc nulls last
        ) as egrn_row_number
    from address_matches m
),

one_row_per_egrn_object as (
    select r.*
    from ranked_egrn_rows r
    where r.egrn_row_number = 1
),

area_check as (
    /* Площадь проверяем только там, где она есть с обеих сторон. */
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as address_candidate_count,
        case
            when c.sphere_area > 0
             and c.square is not null
             and abs(c.square - c.sphere_area)
                 <= greatest(1, c.sphere_area * 0.01)
                then 1
            else 0
        end as area_matches
    from one_row_per_egrn_object c
),

area_choice as (
    select
        a.*,
        max(a.area_matches) over (
            partition by a.sphere_row_id
        ) as has_area_match
    from area_check a
),

candidates_after_area as (
    /* Если площадь помогла, оставляем совпавших. Иначе никого не отсекаем. */
    select a.*
    from area_choice a
    where a.has_area_match = 0
       or a.area_matches = 1
),

candidate_counts as (
    select
        c.*,
        count(*) over (
            partition by c.sphere_row_id
        ) as candidate_count
    from candidates_after_area c
),

candidate_summary as (
    select
        c.sphere_row_id,
        max(c.candidate_count) as candidate_count,
        max(c.address_candidate_count) as address_candidate_count,
        max(c.has_area_match) as has_area_match
    from candidate_counts c
    group by c.sphere_row_id
),

chosen_egrn as (
    /* Здание ЕГРН присоединяется только при одном кандидате. */
    select c.*
    from candidate_counts c
    where c.candidate_count = 1
)

select
    sphere.sphere_row_id as "sphere_row_id",

    /* Договор и основные ID строки Сферы. */
    sphere.contract_number as "Номер договора",
    sphere.contract_id as "ID договора",
    sphere.task_id as "ID задачи",
    sphere.task_object_link_id as "ID связи задачи и объекта",
    sphere.characteristics_id as "ID характеристик",
    sphere.object_id as "ID объекта Сферы",
    sphere.geo_address_id as "ID адреса Сферы",
    sphere.real_estate_objects_in_contract
        as "Объектов недвижимости в договоре",

    /* Объект и исходные адресные строки. */
    sphere.object_description as "Описание объекта Сферы",
    sphere.full_address as "Полный адрес Сферы",
    sphere.original_address as "Исходный адрес Сферы",
    prepared.source_address as "Адрес, который разбирал запрос",
    sphere.total_area as "Площадь Сферы",

    /* Части адреса, которые уже лежали в отдельных колонках Сферы. */
    sphere.postal_code as "Сфера: почтовый индекс",
    sphere.settlement as "Сфера: населённый пункт",
    sphere.street as "Сфера: улица",
    sphere.house as "Сфера: дом",
    sphere.block as "Сфера: корпус",
    sphere.building as "Сфера: строение",
    sphere.flat as "Сфера: квартира или помещение",
    sphere.office as "Сфера: офис",

    /* Так запрос разбил исходную строку адреса до очистки. */
    prepared.postal_code_raw as "После разбора: почтовый индекс",
    prepared.region_raw as "После разбора: регион",
    prepared.locality_raw as "После разбора: населённый пункт",
    prepared.street_raw as "После разбора: улица",
    prepared.house_raw as "После разбора: дом",
    prepared.korpus_raw as "После разбора: корпус",
    prepared.stroenie_raw as "После разбора: строение",

    /* Итог поиска. */
    case
        when prepared.source_address is null
            then 'В Сфере нет адреса'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Не удалось выделить населённый пункт, улицу или дом'
        when nvl(summary.candidate_count, 0) = 0
            then 'Здание ЕГРН не найдено'
        when summary.candidate_count = 1
            then 'Найдено одно здание ЕГРН'
        else 'Найдено несколько зданий. ЕГРН не присоединён'
    end as "Результат поиска",
    nvl(summary.address_candidate_count, 0)
        as "Кандидатов по адресу",
    nvl(summary.candidate_count, 0)
        as "Кандидатов после площади",
    case
        when summary.candidate_count = 1 then 1
        else 0
    end as "Соединение ЕГРН единичное",
    case
        when prepared.source_address is null
            then 'Нет адреса в Сфере'
        when prepared.sphere_locality is null
          or prepared.sphere_street is null
          or prepared.sphere_house is null
            then 'Адрес не удалось разобрать до здания'
        when nvl(summary.candidate_count, 0) = 0
            then 'Совпадение ЕГРН не найдено'
        when summary.candidate_count > 1
            then 'Неоднозначное совпадение'
        when summary.has_area_match = 1
            then 'Адрес здания и площадь'
        when prepared.sphere_area is null
            then 'Только адрес здания: в Сфере нет площади'
        when chosen.square is null
            then 'Только адрес здания: в ЕГРН нет площади'
        else 'Только адрес здания: площадь не подтвердила совпадение'
    end as "Основание соединения ЕГРН",
    case
        when summary.address_candidate_count > summary.candidate_count
            then 'Да'
        else 'Нет'
    end as "Площадь помогла сузить поиск",

    /* Очищенные значения показаны парами: Сфера и найденное здание КХД. */
    prepared.sphere_postal_code as "Сравнение: индекс Сферы",
    chosen.egrn_postal_code as "Сравнение: индекс КХД",

    prepared.sphere_region as "Сравнение: регион Сферы",
    chosen.egrn_region as "Сравнение: регион КХД",

    prepared.sphere_locality as "Сравнение: населённый пункт Сферы",
    case
        when prepared.sphere_locality = chosen.egrn_city
            then chosen.egrn_city
        when prepared.sphere_locality = chosen.egrn_settlement
            then chosen.egrn_settlement
    end as "Сравнение: населённый пункт КХД",

    prepared.sphere_street as "Сравнение: улица Сферы",
    chosen.egrn_street as "Сравнение: улица КХД",

    prepared.sphere_house as "Сравнение: дом Сферы",
    case
        when prepared.sphere_house = chosen.egrn_house
            then chosen.egrn_house
        when prepared.sphere_house = chosen.egrn_vladenie
            then chosen.egrn_vladenie
    end as "Сравнение: дом КХД",

    prepared.sphere_korpus as "Сравнение: корпус Сферы",
    chosen.egrn_korpus as "Сравнение: корпус КХД",

    prepared.sphere_stroenie as "Сравнение: строение Сферы",
    chosen.egrn_stroenie as "Сравнение: строение КХД",

    /* Поля одного найденного здания. */
    chosen.cad_ind as "Внутренний ID здания ЕГРН",
    chosen.cadaster as "Кадастровый номер здания",
    chosen.egrn_address as "Адрес здания ЕГРН",
    chosen.square as "Площадь здания ЕГРН",
    chosen.measure as "Единица площади здания",
    chosen.building_type as "Тип строения здания ЕГРН",
    chosen.oks_type as "Тип здания ЕГРН",
    chosen.oks_purpose as "Назначение здания ЕГРН",
    chosen.object_status as "Статус здания ЕГРН",
    chosen.fias_level as "Уровень адреса ЕГРН",
    chosen.fias_id_house as "ФИАС дома ЕГРН",
    case
        when chosen.cad_ind is not null then 'Здание'
    end as "Уровень присоединения"

from sphere_source sphere
join sphere_prepared prepared
    on prepared.sphere_row_id = sphere.sphere_row_id
left join candidate_summary summary
    on summary.sphere_row_id = sphere.sphere_row_id
left join chosen_egrn chosen
    on chosen.sphere_row_id = sphere.sphere_row_id
order by
    sphere.contract_number,
    sphere.object_id;

/*
Как читать результат
--------------------
Найдено одно здание ЕГРН
    Здание присоединено ко всем объектам Сферы с этим адресом.

Найдено несколько зданий
    По адресу есть несколько кадастровых зданий. Ничего не присоединено.

Здание ЕГРН не найдено
    Адрес удалось разобрать, но запись уровня FIAS_HOUSE не найдена.

Не удалось выделить населённый пункт, улицу или дом
    Адрес есть, но его недостаточно для безопасного автоматического поиска.
*/

"""


In [ ]:
# выполняем Oracle SQL и получаем одну строку результата на строку Сферы
if not KHD_DATA_SCHEMA.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

egrn_query = egrn_sql.replace(
    'DM_RISK_AVATAR.EGRN_DATA',
    f'{KHD_DATA_SCHEMA.upper()}.EGRN_DATA',
)
egrn_lookup_df = pd.read_sql_query(egrn_query, khd_connection)

egrn_column_names = {
    'Результат поиска': 'egrn_match_status',
    'Кандидатов по адресу': 'egrn_address_candidate_count',
    'Кандидатов после площади': 'egrn_candidate_count',
    'Соединение ЕГРН единичное': 'egrn_is_unique_match',
    'Основание соединения ЕГРН': 'egrn_match_basis',
    'Площадь помогла сузить поиск': 'egrn_area_narrowed_search',
    'Адрес, который разбирал запрос': 'sphere_address_for_egrn',
    'После разбора: почтовый индекс': 'parsed_postal_code',
    'После разбора: регион': 'parsed_region',
    'После разбора: населённый пункт': 'parsed_locality',
    'После разбора: улица': 'parsed_street',
    'После разбора: дом': 'parsed_house',
    'После разбора: корпус': 'parsed_korpus',
    'После разбора: строение': 'parsed_stroenie',
    'Сравнение: населённый пункт КХД': 'matched_egrn_locality',
    'Сравнение: улица КХД': 'matched_egrn_street',
    'Сравнение: дом КХД': 'matched_egrn_house',
    'Сравнение: корпус КХД': 'matched_egrn_korpus',
    'Сравнение: строение КХД': 'matched_egrn_stroenie',
    'Внутренний ID здания ЕГРН': 'egrn_cad_ind',
    'Кадастровый номер здания': 'egrn_cadaster',
    'Адрес здания ЕГРН': 'egrn_address',
    'Площадь здания ЕГРН': 'egrn_square',
    'Единица площади здания': 'egrn_measure',
    'Тип строения здания ЕГРН': 'egrn_building_type',
    'Тип здания ЕГРН': 'egrn_oks_type',
    'Назначение здания ЕГРН': 'egrn_oks_purpose',
    'Статус здания ЕГРН': 'egrn_object_status',
    'Уровень адреса ЕГРН': 'egrn_fias_level',
    'ФИАС дома ЕГРН': 'egrn_fias_id_house',
    'Уровень присоединения': 'egrn_join_level',
}
egrn_lookup_df = egrn_lookup_df.rename(columns=egrn_column_names)
egrn_columns = ['sphere_row_id'] + list(egrn_column_names.values())

missing_egrn_columns = [
    column for column in egrn_columns
    if column not in egrn_lookup_df.columns
]
if missing_egrn_columns:
    raise ValueError(
        'Oracle не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('Oracle вернул несколько строк для одного объекта Сферы')

expanded_egrn_df = sphere_with_row_id.merge(
    egrn_lookup_df[egrn_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

if len(expanded_egrn_df) != len(sphere_with_row_id):
    raise ValueError('После соединения с ЕГРН изменилось количество строк')

print('Строк после соединения с ЕГРН:', len(expanded_egrn_df))


# 6. Проверка соединения с ЕГРН


In [ ]:
egrn_profile = (
    expanded_egrn_df['egrn_match_status']
    .fillna('Нет результата Oracle')
    .value_counts(dropna=False)
    .rename_axis('Результат поиска')
    .reset_index(name='Количество строк')
)
display(egrn_profile)

egrn_preview_columns = [
    'contract_id',
    'object_id',
    'full_address',
    'total_area',
    'egrn_match_status',
    'egrn_is_unique_match',
    'egrn_match_basis',
    'egrn_candidate_count',
    'egrn_cadaster',
    'egrn_address',
    'egrn_square',
]
display(expanded_egrn_df[egrn_preview_columns].head(20))


# 7. Сохранение результата


In [40]:
output_path = OUTPUT_DIR / 'датасет_расширенный_с_егрн.csv'
expanded_egrn_df.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print('Сохранено:', output_path)


Сохранено: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\данные\датасет_51_расширенный.csv


# 8. Сохранение списка ИНН

Отдельный файл содержит только одну колонку `inn`. Пустые значения и повторы исключаются.


In [ ]:
inn_df = (
    expanded_egrn_df[['policyholder_inn']]
    .rename(columns={'policyholder_inn': 'inn'})
    .assign(inn=lambda data: data['inn'].astype('string').str.strip())
    .loc[lambda data: data['inn'].notna() & data['inn'].ne('')]
    .drop_duplicates(subset=['inn'])
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(inn_path, index=False, sep=';', encoding='utf-8-sig')
print('Уникальных ИНН:', len(inn_df))
print('Сохранено:', inn_path)


In [41]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')

Подключения к Сфере и КХД закрыты
